# Standalone APUSH HF GPU Eval

This notebook does **not** require running from the local `slm/` repo. It is meant for Colab / notebook GPU execution.

It does four things:
1. Pulls the APUSH eval prompt and source data from GitHub.
2. Loads the Qwen base model and APUSH LoRA adapter from Hugging Face.
3. Calls the teacher and judge through the API token.
4. Runs the same base-vs-tuned-vs-teacher litmus protocol inline in the notebook.

Default models:
- base: `unsloth/Qwen3-4B`
- tuned adapter: `rohanpalviela/qwen3-4b-apush-v4-semantic-audited-lora`
- teacher: `claude-group/claude-opus-4-8`
- judge: `openai-group/gpt-5.5`

In [ ]:
# Required: paste the full 40-character commit SHA, then run this cell first.
import os
os.environ['APUSH_GITHUB_REF'] = 'REPLACE_WITH_FULL_40_CHARACTER_COMMIT_SHA'
print('APUSH_GITHUB_REF:', os.environ['APUSH_GITHUB_REF'])

In [ ]:
# Runtime setup. Run this first.
import hashlib, json, math, os, random, re, subprocess, sys, time, urllib.error, urllib.parse, urllib.request
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path

WORKDIR = Path('/content/apush_eval') if Path('/content').exists() else Path.cwd() / 'apush_eval_workspace'
WORKDIR.mkdir(parents=True, exist_ok=True)
print('workdir:', WORKDIR)
print('python:', sys.executable)

## Configuration

Set `PROMPTLENS_API_KEY` for teacher generation and judging. Set `HF_TOKEN` only if the base or adapter repo is private/gated.

Set `GITHUB_REF` to the full 40-character commit SHA containing this notebook and the evaluator modules. Branch names are rejected.

In [ ]:
# Data/prompt source
GITHUB_OWNER_REPO = 'RohanPalivela/slm'
GITHUB_REF = os.environ.get('APUSH_GITHUB_REF', 'REPLACE_WITH_FULL_COMMIT_SHA')
if not re.fullmatch(r'[0-9a-f]{40}', GITHUB_REF):
    raise RuntimeError('Set APUSH_GITHUB_REF to the full 40-character commit SHA before GPU evaluation.')
RAW_BASE = f'https://raw.githubusercontent.com/{GITHUB_OWNER_REPO}/{GITHUB_REF}'

# HF local inference
BASE_MODEL_ID = 'unsloth/Qwen3-4B'
BASE_MODEL_REVISION = '64033659d5caf1b8ed7f929b29de705e93a4d468'  # exact immutable training base
APUSH_ADAPTER_ID = 'rohanpalviela/qwen3-4b-apush-v4-semantic-audited-lora'
EXPECTED_ADAPTER_TRAINING_DATA_SHA256 = 'fb49a00b5edb413cfd004f398261cc409a193da6d1afd8a2187166b206a8e608'
LOAD_IN_4BIT = False  # faster on L4/A100; set True only if you hit OOM
MAX_NEW_TOKENS = 768    # smoke/eval speed lever; raise to 1536 if outputs truncate
TEMPERATURE = 0.2       # low-variance sampling with deterministic per-trial seeds
MAX_CONTRACT_ATTEMPTS = 8  # first generation plus bounded mechanical-contract retries
EVAL_SEED = 20260710    # matched HF repetition batches keyed by source and archetype
# The production generation protocol is hardcoded where HFLocalEngine is constructed.
# Protocol experiments belong in a separate generation-only notebook, never this semantic eval.
DOWNLOAD_RESULTS_ZIP = True
PERSIST_CHECKPOINTS_TO_DRIVE = True  # survives Colab runtime loss after Drive authorization
CHECKPOINT_DIR_ENV = 'APUSH_EVAL_CHECKPOINT_DIR'  # optional explicit persistent path
EVAL_EXECUTION_VERSION = 'conditional-valid-pair-exclusion-v6'  # invalidates abort-on-exhaustion checkpoints

# API models
API_KEY_ENV = 'PROMPTLENS_API_KEY'
API_BASE_URL = 'https://tfy.promptlens.trilogy.com/v1'
TEACHER_MODEL = 'claude-group/claude-opus-4-8'
JUDGE_MODEL = 'openai-group/gpt-5.5'

# Eval scope
SPLIT = 'EVAL_HELDOUT'
PREFLIGHT_SPLIT = 'LITMUS'
# Frozen 14-source cohort: proportional genre coverage plus broad chronological coverage.
REPRESENTATIVE_EVAL_SOURCE_IDS = [
    'federalist_10_1787', 'us_constitution_1787', 'marbury_madison_1803',
    'indian_removal_act_1830', 'treaty_guadalupe_hidalgo_1848', 'dred_scott_1857',
    'fourteenth_amendment_1868', 'chief_joseph_1877', 'clayton_antitrust_act_1914',
    'schenck_us_1919', 'social_security_act_1935', 'executive_order_9066_1942',
    'executive_order_9981_1948', 'civil_rights_act_1964',
]
ARCHETYPES = ['CAUSE_OF_SOURCE', 'EFFECT_OF_SOURCE']
DIFFICULTY = 'operational / test-day'
INCLUDE_FEWSHOT = False

# Optional: read tokens from Colab Secrets if available.
try:
    from google.colab import userdata
    for _name in ['PROMPTLENS_API_KEY', 'HF_TOKEN']:
        if not os.environ.get(_name):
            try:
                _value = userdata.get(_name)
            except Exception:
                _value = None
            if _value:
                os.environ[_name] = _value
except Exception:
    pass

# Candidates retain two matched low-temperature samples over the representative cohort.
# The teacher is a one-pass reference because repeated teacher calls do not affect the paired candidate test.
RUNS = 2
TEACHER_RUNS = 1
API_MAX_WORKERS = 6  # bounded concurrency for teacher generation and judging
CHECKPOINT_FLUSH_EVERY = 6  # amortize persistent-drive writes while limiting interruption loss
N = 2           # requested items per source; notebook calls once per archetype
RUN_TEACHER = bool(os.environ.get(API_KEY_ENV))
USE_JUDGE = bool(os.environ.get(API_KEY_ENV))

print('teacher enabled:', RUN_TEACHER, 'runs:', TEACHER_RUNS if RUN_TEACHER else 0)
print('judge enabled:', USE_JUDGE, 'API workers:', API_MAX_WORKERS if USE_JUDGE else 0)
if not os.environ.get(API_KEY_ENV):
    print(f'No {API_KEY_ENV}: base+tuned can still run with programmatic checks, but teacher/judge are disabled.')

## Install GPU Inference Dependencies

On Colab, use a GPU runtime before running this. Keep `LOAD_IN_4BIT=False` on L4/A100 unless you hit OOM; fp16/bf16 is usually faster than bitsandbytes 4-bit for this 4B model.

In [ ]:
DEPENDENCY_SPECS = [
    'transformers', 'accelerate', 'peft', 'bitsandbytes', 'sentencepiece',
    'huggingface_hub', 'torchao>=0.16.0',
]
DEPENDENCY_PROBE = """
import importlib.metadata as metadata
from packaging.version import Version
import huggingface_hub._snapshot_download
from transformers import AutoTokenizer
import accelerate, bitsandbytes, peft, torchao
assert Version(metadata.version('torchao')) >= Version('0.16.0')
"""
def fresh_dependency_probe():
    return subprocess.run([sys.executable, '-c', DEPENDENCY_PROBE], text=True, capture_output=True)

fresh_probe = fresh_dependency_probe()
kernel_probe_error = None
try:
    exec(DEPENDENCY_PROBE, {})
except Exception as exc:
    kernel_probe_error = repr(exc)

installed_or_repaired = False
if fresh_probe.returncode != 0:
    cmd = [sys.executable, '-m', 'pip', 'install', '-q', '-U', *DEPENDENCY_SPECS]
    print('+', ' '.join(cmd))
    subprocess.check_call(cmd)
    # Repair mixed on-disk hub files left by interrupted or overlapping Colab installs.
    import importlib.metadata as dependency_metadata
    resolved_hub_version = dependency_metadata.version('huggingface_hub')
    hub_repair = [sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir', '--force-reinstall', '--no-deps', f'huggingface_hub=={resolved_hub_version}']
    print('+', ' '.join(hub_repair))
    subprocess.check_call(hub_repair)
    fresh_probe = fresh_dependency_probe()
    if fresh_probe.returncode != 0:
        raise RuntimeError(f'Dependency installation is still inconsistent:\n{fresh_probe.stderr[-4000:]}')
    installed_or_repaired = True

if installed_or_repaired or kernel_probe_error is not None:
    reason = 'dependencies changed' if installed_or_repaired else f'current kernel has stale imports: {kernel_probe_error}'
    print(f'{reason}. Restarting the runtime once; after reconnection, run the notebook from the top again.')
    try:
        get_ipython().kernel.do_shutdown(restart=True)
    finally:
        raise SystemExit('Runtime restart required after dependency repair.')

import importlib.metadata as importlib_metadata
print('dependency versions:', {name: importlib_metadata.version(name) for name in ['transformers', 'accelerate', 'peft', 'bitsandbytes', 'huggingface_hub', 'torchao']})

import torch
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
    print('capability:', torch.cuda.get_device_capability(0))
else:
    print('WARNING: no CUDA GPU. This will be very slow.')


## Pull Prompt and Eval Data

This pulls only the files needed for eval. It does not clone or import the local repo.

In [ ]:
def fetch_text(path, *, required=True):
    url = f'{RAW_BASE}/{path}'
    try:
        with urllib.request.urlopen(url, timeout=30) as r:
            text = r.read().decode('utf-8')
    except Exception as e:
        if required:
            raise RuntimeError(f'Failed to fetch {url}. If this branch is wrong, set GITHUB_REF.') from e
        return None
    out = WORKDIR / path
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text(text, encoding='utf-8')
    print('fetched', path, f'({len(text):,} chars)')
    return text

prompt_md = fetch_text('prompts/litmus_generation_prompt.md')
splits = json.loads(fetch_text('data/splits.json'))
seed_stimuli_text = fetch_text('data/seed_stimuli.jsonl')
theme_vocabulary_text = fetch_text('data/apush_periods_themes.json')
canonical_notebook_text = fetch_text('notebooks/eval_hf_gpu.ipynb')
seed_stimuli_lines = seed_stimuli_text.splitlines()
EVALUATOR_PATHS = ['eval/date_utils.py', 'eval/checks.py', 'eval/prompt_loader.py', 'eval/judge.py', 'eval/providers.py', 'eval/source_utils.py', 'eval/notebook_runtime.py', 'eval/hf_local.py']
for shared_path in EVALUATOR_PATHS:
    fetch_text(shared_path)
sys.path.insert(0, str(WORKDIR / 'eval'))
seed_stimuli = [json.loads(line) for line in seed_stimuli_lines if line.strip()]
print('sources:', len(seed_stimuli))
print('splits:', ', '.join(splits['splits']))

## Inline Prompt Loader, JSON Parser, and Programmatic Checks

In [ ]:
def fenced_block_after(md_text, header_substr):
    lines = md_text.splitlines(); last_header = ''; i = 0
    while i < len(lines):
        line = lines[i]
        if line.lstrip().startswith('#'): last_header = line.lower()
        if line.lstrip().startswith('```'):
            body = []; i += 1
            while i < len(lines) and not lines[i].lstrip().startswith('```'):
                body.append(lines[i]); i += 1
            if header_substr.lower() in last_header: return '\n'.join(body)
        i += 1
    return None

@dataclass
class LitmusPrompt:
    system: str
    user: str
    fewshot: str | None = None
    @classmethod
    def from_markdown(cls, md_text):
        system = fenced_block_after(md_text, '## system')
        user = fenced_block_after(md_text, '## user')
        fewshot = fenced_block_after(md_text, 'few-shot')
        if not system or not user: raise ValueError('could not parse SYSTEM/USER fenced blocks')
        return cls(system.strip(), user.strip(), fewshot.strip() if fewshot else None)
    def build(self, *, source, attribution, note, n, archetypes, difficulty, include_fewshot=False):
        system = self.system
        if include_fewshot and self.fewshot: system += '\n\nFEW-SHOT EXEMPLARS:\n' + self.fewshot
        user = (self.user.replace('{{SOURCE}}', source).replace('{{ATTRIBUTION}}', attribution)
                .replace('{{NOTE}}', note or '(none)').replace('{{N}}', str(n))
                .replace('{{ARCHETYPES}}', archetypes).replace('{{DIFFICULTY}}', difficulty))
        return system, user

ITEM_LIST_KEYS = ('items', 'questions', 'mcqs', 'data', 'results', 'output')
ARCHETYPE_ALIASES = {'cause_of':'CAUSE_OF_SOURCE','effect_immediate':'EFFECT_OF_SOURCE','effect_longterm':'LONGTERM_LEGACY','continuation_change':'CONTINUITY_OR_CHANGE','reflects_illustrates':'DEVELOPMENT_ILLUSTRATED','context_response_to':'CONTEXT_SITUATION','influenced_by':'CONTEXT_INFLUENCED_BY','purpose_intended_to':'SOURCE_POV_PURPOSE','point_of_view':'SOURCE_POV_PURPOSE','evidence_supports':'EVIDENCE_SUPPORTS_CLAIM','evidence_undermines':'EVIDENCE_UNDERMINES_CLAIM','similar_effect':'COMPARATIVE_ANALOG','differs_from':'COMPETING_INTERPRETATIONS'}

def normalize_items(obj):
    if isinstance(obj, dict):
        for key in ITEM_LIST_KEYS:
            if isinstance(obj.get(key), list): return normalize_items(obj[key])
        return [obj]
    if isinstance(obj, list): return [x for x in obj if isinstance(x, dict)]
    return []

def extract_items(text):
    if not text: return []
    t = re.sub(r'\s*```$', '', re.sub(r'^```[a-zA-Z]*\s*', '', text.strip()))
    try: return normalize_items(json.loads(t))
    except json.JSONDecodeError: pass
    for open_ch, close_ch in (('[', ']'), ('{', '}')):
        start = t.find(open_ch)
        if start == -1: continue
        depth = 0; in_str = False; esc = False
        for j, c in enumerate(t[start:], start):
            if in_str:
                if esc: esc = False
                elif c == '\\': esc = True
                elif c == '"': in_str = False
                continue
            if c == '"': in_str = True
            elif c == open_ch: depth += 1
            elif c == close_ch:
                depth -= 1
                if depth == 0:
                    try: return normalize_items(json.loads(t[start:j+1]))
                    except json.JSONDecodeError: break
    return []

def canonicalize_item_archetype(item, requested_archetype=None):
    out = dict(item); raw = out.get('archetype')
    if raw is not None and '_model_archetype' not in out: out['_model_archetype'] = raw
    if requested_archetype:
        out['archetype'] = requested_archetype; out['_requested_archetype'] = requested_archetype
    elif isinstance(raw, str): out['archetype'] = ARCHETYPE_ALIASES.get(raw.strip(), raw)
    return out

ABSOLUTE_OR_ALLNONE = re.compile(r'\b(all|none)\s+of\s+the\s+above\b|\balways\b|\bnever\b', re.I)
YEAR_RE = re.compile(r'\b(1[4-9]\d\d|20\d\d)s?\b')
PARENTHETICAL_DATE_LABEL = re.compile(r'\(\s*(?:c\.\s*)?(?:1[4-9]\d\d|20\d\d)(?:\s*[-–]\s*(?:c\.\s*)?(?:\d{2,4}|present))?\s*\)', re.I)
VALID_TRAP_TYPES = {'WRONG_ERA','TRUE_BUT_IRRELEVANT','SCOPE_MISMATCH','PARTIALLY_TRUE'}
CAUSE_ARCHETYPES = {'CAUSE_OF_SOURCE','CONTEXT_SITUATION','CONTEXT_INFLUENCED_BY'}
EFFECT_ARCHETYPES = {'EFFECT_OF_SOURCE','LONGTERM_LEGACY','COMPARATIVE_ANALOG','CONTINUITY_OR_CHANGE'}

def strip_label(opt): return re.sub(r'^\s*[A-D][).:]?\s+', '', opt or '').strip()
def answer_index(item):
    a = str(item.get('answer', '')).strip().upper()
    return 'ABCD'.index(a[0]) if a and a[0] in 'ABCD' else None
def norm_text(s): return re.sub(r'\s+', ' ', (s or '').lower()).strip()
def years(text): return [int(y) for y in YEAR_RE.findall(text or '')]
def source_year(source):
    y = source.get('year')
    if isinstance(y, list) and y: return int(y[0])
    if isinstance(y, int): return y
    found = years(source.get('attribution', '')); return found[0] if found else None

def date_direction(item, source):
    src_year = source_year(source); arch = item.get('archetype', '')
    if src_year is None or (arch not in CAUSE_ARCHETYPES and arch not in EFFECT_ARCHETYPES): return 'unknown'
    ys = [y for y in years(item.get('answer_dating', '')) if y != src_year]
    if not ys: return 'unknown'
    return ('fail' if min(ys) > src_year else 'pass') if arch in CAUSE_ARCHETYPES else ('fail' if max(ys) < src_year else 'pass')

def source_leak(item, source):
    idx = answer_index(item); opts = item.get('options', [])
    if idx is None or idx >= len(opts): return False
    ans = norm_text(strip_label(opts[idx])); return len(ans) >= 40 and ans in norm_text(source.get('text', ''))

def rationale_complete(item):
    rat = item.get('rationale'); return isinstance(rat, dict) and all(k in rat and str(rat.get(k, '')).strip() for k in 'ABCD')
def rationale_marks_key(item):
    idx = answer_index(item); rat = item.get('rationale')
    if idx is None or not isinstance(rat, dict): return False
    txt = str(rat.get('ABCD'[idx], '')).strip().lower()
    if txt in {'correct','correct.'} or txt.startswith('correct:'): return True
    return bool(txt) and not any(txt.startswith(t.lower()) for t in VALID_TRAP_TYPES)

def run_checks(item, source):
    opts = item.get('options', []); idx = answer_index(item); labels = [strip_label(o) for o in opts] if opts else []
    four_options = isinstance(opts, list) and len(opts) == 4
    one_key = idx is not None and 0 <= idx < len(opts)
    no_all_none = not any(ABSOLUTE_OR_ALLNONE.search(o or '') for o in opts)
    homogeneous = True
    if four_options and one_key and labels:
        lens = sorted(len(l) for k, l in enumerate(labels) if k != idx); homogeneous = len(labels[idx]) <= 1.7 * (lens[len(lens)//2] or 1)
    traps = [str(t).strip().upper() for t in (item.get('trap_types', []) or []) if str(t).strip()]
    trap_count_3 = len(traps) == 3; trap_types_valid = bool(traps) and all(t in VALID_TRAP_TYPES for t in traps)
    trap_diversity = len(set(t for t in traps if t != 'CORRECT')) >= 2; wrong_era_le1 = sum(1 for t in traps if t == 'WRONG_ERA') <= 1
    no_option_dates = not any(PARENTHETICAL_DATE_LABEL.search(o or '') for o in opts)
    leak = source_leak(item, source); date = date_direction(item, source); schema_ok = rationale_complete(item) and rationale_marks_key(item)
    disqualifying_ok = four_options and one_key and no_all_none and not leak and date != 'fail'
    craft_ok = trap_count_3 and trap_types_valid and trap_diversity and wrong_era_le1 and no_option_dates and schema_ok
    return {'four_options': four_options, 'one_key': one_key, 'no_all_none_absolute': no_all_none, 'homogeneous_length': homogeneous,
            'trap_count_3': trap_count_3, 'trap_types_valid': trap_types_valid, 'trap_diversity_ge2': trap_diversity,
            'wrong_era_le1': wrong_era_le1, 'no_parenthetical_option_dates': no_option_dates, 'rationale_complete': rationale_complete(item),
            'rationale_marks_key': rationale_marks_key(item), 'source_leak': leak, 'date_direction': date,
            'disqualifying_ok': disqualifying_ok, 'schema_ok': schema_ok, 'craft_ok': craft_ok}

prompt = LitmusPrompt.from_markdown(prompt_md)
by_id = {s['id']: s for s in seed_stimuli}
heldout_source_ids = splits['splits'][SPLIT]['source_ids']
outside_heldout = [sid for sid in REPRESENTATIVE_EVAL_SOURCE_IDS if sid not in heldout_source_ids]
if outside_heldout: raise RuntimeError(f'representative eval ids are not in {SPLIT}: {outside_heldout}')
source_ids = list(REPRESENTATIVE_EVAL_SOURCE_IDS)
if len(source_ids) != 14 or len(set(source_ids)) != 14: raise RuntimeError('representative eval cohort must contain 14 unique sources')
missing_source_ids = [sid for sid in source_ids if sid not in by_id]
if missing_source_ids: raise RuntimeError(f'split references missing source ids: {missing_source_ids}')
sources = [by_id[sid] for sid in source_ids]
# Override notebook-local copies with the repository's canonical evaluator.
# This prevents date/check behavior from drifting between CLI and Colab.
import checks as shared_checks
from prompt_loader import extract_items as shared_extract_items, canonicalize_item_archetype as shared_canonicalize, generation_format_diagnostics
from source_utils import source_genre
from notebook_runtime import aggregate_attempt_metrics, append_jsonl_group, attempt_contract_outcome, attempt_key, attempt_seed, dedupe_rows, load_jsonl_groups, matched_candidate_exclusions, runs_for_model, score_key, source_clustered_paired_ci, stable_hash
run_checks = shared_checks.run_checks
extract_items = shared_extract_items
canonicalize_item_archetype = shared_canonicalize
print('loaded split:', SPLIT, 'sources:', len(sources), [s['id'] for s in sources[:5]])

## Providers: HF Local GPU + API Teacher/Judge

The HF engine loads the base model once, attaches the APUSH adapter once, and uses `disable_adapter()` for base generations. That avoids reloading a 4B model for every model comparison.

In [ ]:
class ProviderError(Exception): pass

def post_json(url, headers, payload, timeout=240):
    data = json.dumps(payload).encode('utf-8')
    req = urllib.request.Request(url, data=data, headers=headers, method='POST')
    last = None
    for attempt in range(3):
        try:
            with urllib.request.urlopen(req, timeout=timeout) as r: return json.load(r)
        except urllib.error.HTTPError as e:
            body = e.read().decode('utf-8', 'replace'); last = ProviderError(f'HTTP {e.code}: {body[:800]}')
            if e.code in (429,500,502,503,529): time.sleep(2*(attempt+1)); continue
            raise last
        except Exception as e:
            last = ProviderError(f'network error: {e}'); time.sleep(2*(attempt+1))
    raise last or ProviderError('unknown API error')

def api_generate(model, system, user, *, max_tokens=4096, temperature=0.7, include_temperature=True, token_param='max_tokens'):
    key = os.environ.get(API_KEY_ENV, '')
    if not key: raise ProviderError(f'missing {API_KEY_ENV}')
    payload = {'model': model, 'messages': [{'role':'system','content':system}, {'role':'user','content':user}], token_param: max_tokens}
    if include_temperature: payload['temperature'] = temperature
    headers = {'Content-Type':'application/json', 'Authorization':f'Bearer {key}'}
    try: resp = post_json(API_BASE_URL.rstrip() + '/chat/completions', headers, payload)
    except ProviderError as e:
        s = str(e).lower()
        if 'http 400' in s and 'temperature' in s and include_temperature:
            return api_generate(model, system, user, max_tokens=max_tokens, temperature=temperature, include_temperature=False, token_param=token_param)
        if 'http 400' in s and 'max_tokens' in s and token_param == 'max_tokens':
            return api_generate(model, system, user, max_tokens=max_tokens, temperature=temperature, include_temperature=include_temperature, token_param='max_completion_tokens')
        raise
    return ((resp.get('choices') or [{}])[0].get('message') or {}).get('content') or ''

from hf_local import HFLocalEngine
hf_engine = None

## Check HF Access and Chat Template

In [ ]:
from huggingface_hub import hf_hub_download, model_info
hf_token = os.environ.get('HF_TOKEN') or None
base_info = model_info(BASE_MODEL_ID, revision=BASE_MODEL_REVISION, token=hf_token)
adapter_info = model_info(APUSH_ADAPTER_ID, token=hf_token)
if not re.fullmatch(r'[0-9a-f]{40}', adapter_info.sha or ''):
    raise RuntimeError(f'Adapter revision is not immutable: {adapter_info.sha!r}')
MODEL_REVISIONS = {BASE_MODEL_ID: BASE_MODEL_REVISION, APUSH_ADAPTER_ID: adapter_info.sha}
print('base revision:', BASE_MODEL_REVISION)
print('adapter revision:', adapter_info.sha)
training_metadata_path = hf_hub_download(APUSH_ADAPTER_ID, 'training_run_metadata.json', revision=adapter_info.sha, token=hf_token)
TRAINING_RUN_METADATA = json.loads(Path(training_metadata_path).read_text(encoding='utf-8'))
TRAINING_RUN_METADATA_SHA256 = hashlib.sha256(Path(training_metadata_path).read_bytes()).hexdigest()
ADAPTER_TRAINING_DATA_SHA256 = TRAINING_RUN_METADATA.get('dataset_sha256')
if not isinstance(ADAPTER_TRAINING_DATA_SHA256, str) or not re.fullmatch(r'[0-9a-f]{64}', ADAPTER_TRAINING_DATA_SHA256):
    raise RuntimeError('Adapter training_run_metadata.json is missing a valid dataset_sha256; semantic comparison is blocked.')
if ADAPTER_TRAINING_DATA_SHA256 != EXPECTED_ADAPTER_TRAINING_DATA_SHA256:
    raise RuntimeError(f'Adapter was trained on the wrong dataset: expected {EXPECTED_ADAPTER_TRAINING_DATA_SHA256}, found {ADAPTER_TRAINING_DATA_SHA256}.')
if TRAINING_RUN_METADATA.get('use_audited_data') is not True:
    raise RuntimeError('Adapter metadata does not confirm use_audited_data=true; semantic comparison is blocked.')
if TRAINING_RUN_METADATA.get('base_model') != BASE_MODEL_ID or TRAINING_RUN_METADATA.get('base_revision') != BASE_MODEL_REVISION:
    raise RuntimeError('Adapter training metadata does not match the configured immutable base model and revision.')
tokenizer_names = [s.rfilename for s in base_info.siblings if Path(s.rfilename).name in {'tokenizer.json','tokenizer_config.json','special_tokens_map.json','added_tokens.json','vocab.json','merges.txt','chat_template.jinja'}]
TOKENIZER_FILE_SHA256 = {}
for name in sorted(tokenizer_names):
    path = hf_hub_download(BASE_MODEL_ID, name, revision=BASE_MODEL_REVISION, token=hf_token)
    TOKENIZER_FILE_SHA256[name] = hashlib.sha256(Path(path).read_bytes()).hexdigest()
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(BASE_MODEL_ID, revision=BASE_MODEL_REVISION, token=hf_token, trust_remote_code=True)
msgs = [{'role':'system','content':'Return only JSON.'}, {'role':'user','content':'Say OK as a JSON array.'}]
try: rendered = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
except TypeError: rendered = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
empty_think = bool(re.search(r'<think>\s*</think>', rendered, re.I | re.S))
nonempty_think = any(span.strip() for span in re.findall(r'<think>(.*?)</think>', rendered, flags=re.I | re.S))
rendered_for_generation = rendered
CHAT_TEMPLATE_SHA256 = hashlib.sha256((tok.chat_template or '').encode('utf-8')).hexdigest()
CANONICAL_NOTEBOOK_SHA256 = hashlib.sha256(canonical_notebook_text.encode('utf-8')).hexdigest()
canonical_notebook = json.loads(canonical_notebook_text)
def canonical_cell_sha256(function_name):
    definition = re.compile(rf'(?m)^def\s+{re.escape(function_name)}\s*\(')
    matches = [''.join(cell.get('source', [])) for cell in canonical_notebook['cells'] if definition.search(''.join(cell.get('source', [])))]
    if len(matches) != 1: raise RuntimeError(f'Expected one canonical notebook cell defining {function_name!r}, found {len(matches)}')
    return hashlib.sha256(matches[0].encode('utf-8')).hexdigest()
CRITICAL_GENERATION_CELL_SHA256 = canonical_cell_sha256('generate_model_repetitions')
PREFLIGHT_CELL_SHA256 = canonical_cell_sha256('run_generation_preflight')
PACKAGE_VERSIONS = {}
for package in ['torch','transformers','peft','accelerate','bitsandbytes','huggingface_hub','torchao']:
    try: PACKAGE_VERSIONS[package] = importlib_metadata.version(package)
    except importlib_metadata.PackageNotFoundError: PACKAGE_VERSIONS[package] = None
hf_engine = HFLocalEngine(base_model_id=BASE_MODEL_ID, base_model_revision=BASE_MODEL_REVISION, adapter_id=APUSH_ADAPTER_ID, adapter_revision=adapter_info.sha, load_in_4bit=LOAD_IN_4BIT, keep_no_think_prefill=True, force_json_array_prefix=False, use_no_think_soft_switch=False, stopping_enabled=True)
print('--- tokenizer-rendered prompt ---')
print(rendered[:1200])
print('--- prompt actually used for generation ---')
print(rendered_for_generation[:1200])
print('empty no-think prefill:', empty_think)
print('nonempty thinking text:', nonempty_think)
print('generation protocol: native no-think prefill, no forced prefix, no /no_think injection, JSON stopping enabled')
print('contains tool scaffold:', '<tools>' in rendered or 'tool_call' in rendered)
print('tokenizer files:', TOKENIZER_FILE_SHA256)
print('adapter training dataset sha256:', ADAPTER_TRAINING_DATA_SHA256)


## Judge Rubric

In [ ]:
JUDGE_SYSTEM = """You are a senior AP U.S. History assessment expert grading a single machine-generated stimulus-based multiple-choice question. You know U.S. history well and you are NOT the model that wrote the item.

Grade like a real test-development reviewer: be STRICT about correctness (is the keyed answer right, is it uniquely best, does the skill match the command phrase) but FAIR about craft. An item a real APUSH exam would actually use is expert-grade even if a distractor could be marginally stronger. Do NOT invent flaws or hold items to a standard of perfection the operational test itself does not meet.

You will receive the SOURCE (stimulus) and the ITEM (stem, 4 options, keyed answer, rationale). Judge ONLY against the source + standard APUSH knowledge.

CALIBRATION:
- distractors_period_plausible is TRUE unless a distractor is genuinely absurd, fabricated, or off-topic filler.
- spec_adherence: 2 = operationally usable; 1 = genuine craft weakness short of disqualifying; 0 = disqualifying flaw.

Return ONLY a JSON object with these fields:
{"requires_outside_knowledge": true|false, "every_distractor_named_trap": true|false, "distractors_period_plausible": true|false, "skill_matches_command_phrase": true|false, "key_historically_correct": true|false, "key_uniquely_best": true|false, "single_best_answer": true|false, "spec_adherence": 0|1|2, "distractor_craft": 0|1|2, "outside_knowledge_skill_fit": 0|1|2, "notes": "<one sentence>"}"""

def fmt_options(opts): return '\n'.join(f"  {'ABCD'[i] if i < 4 else '?'}) {o}" for i, o in enumerate(opts or []))
def build_judge_prompt(source, item):
    user = f"""SOURCE ({source.get('attribution', '')}):\n\"\"\"\n{source.get('text', '')}\n\"\"\"\n\nITEM:\narchetype: {item.get('archetype', '')}\nstem: {item.get('stem', '')}\noptions:\n{fmt_options(item.get('options', []))}\nkeyed answer: {item.get('answer', '')}\nanswer_dating: {item.get('answer_dating', '')}\nrationale: {json.dumps(item.get('rationale', {}), ensure_ascii=False)}\n\nGrade it. Return ONLY the JSON object."""
    return JUDGE_SYSTEM, user

def normalize_judgment(j):
    def b(k): return bool(j.get(k, False))
    def g(k):
        try: return int(j.get(k, 0))
        except Exception: return 0
    out = {'requires_outside_knowledge':b('requires_outside_knowledge'),'every_distractor_named_trap':b('every_distractor_named_trap'),'distractors_period_plausible':b('distractors_period_plausible'),'skill_matches_command_phrase':b('skill_matches_command_phrase'),'key_historically_correct':b('key_historically_correct'),'key_uniquely_best':b('key_uniquely_best'),'single_best_answer':b('single_best_answer'),'spec_adherence':g('spec_adherence'),'distractor_craft':g('distractor_craft'),'outside_knowledge_skill_fit':g('outside_knowledge_skill_fit'),'notes':str(j.get('notes',''))[:300]}
    out['key_valid'] = out['key_historically_correct'] and out['key_uniquely_best']; return out

def judge_item(source, item):
    if not USE_JUDGE: return None
    system, user = build_judge_prompt(source, item)
    raw = api_generate(JUDGE_MODEL, system, user, max_tokens=2048, temperature=0.0, include_temperature=False, token_param='max_completion_tokens')
    parsed = extract_items(raw)
    return normalize_judgment(parsed[0] if parsed else {'notes':'judge returned unparseable output'})

def near_grade(prog_ok, j):
    if j is None: return None
    judge_disq = (j['requires_outside_knowledge'] and j['every_distractor_named_trap'] and j['distractors_period_plausible'] and j['skill_matches_command_phrase'] and j['single_best_answer'] and j['key_valid'])
    dims_ok = min(j['spec_adherence'], j['distractor_craft'], j['outside_knowledge_skill_fit']) >= 1
    return bool(prog_ok and judge_disq and dims_ok)
def expert_grade(prog_ok, j):
    ng = near_grade(prog_ok, j); return None if ng is None else bool(ng and j['spec_adherence'] == 2)

# Canonical shared rubric, retry logic, raw-response capture, and grading gates.
import judge as shared_judge
JUDGE_SYSTEM = shared_judge.JUDGE_SYSTEM
build_judge_prompt = shared_judge.build_judge_prompt
normalize_judgment = shared_judge._normalize
near_grade = shared_judge.near_grade
expert_grade = shared_judge.expert_grade
JUDGE_CONFIG = {'name':'eval-judge','provider':'openai_compatible','model':JUDGE_MODEL,'base_url':API_BASE_URL,'api_key_env':API_KEY_ENV,'max_tokens':2048,'include_temperature':False,'token_param':'max_completion_tokens','parse_attempts':3}
def judge_item(source, item):
    if not USE_JUDGE: return None
    return shared_judge.judge_item(JUDGE_CONFIG, source, item, role='evaluation')

## Generation-Only LITMUS Preflight

This loads the HF model and adapter, then checks all 20 LITMUS source-archetype prompts for both base and tuned paths.
Execution failures and thinking-tag protocol regressions abort before API calls.
Other product-contract failures are recorded as diagnostics because the full evaluation applies the same bounded contract-valid retry policy to both candidates.

In [ ]:
SCHEMA_VALIDATION_KEYS = (
    'required_fields_present', 'field_types_exact', 'four_string_options',
    'answer_key_valid', 'rationale_mapping_complete', 'rationale_correct_present',
    'rationale_marks_answer', 'archetype_matches_request', 'period_matches_source',
    'theme_valid', 'nonempty_string_fields', 'trap_types_are_strings',
)
def contract_constrained_user(user, source, arch, validation=None):
    retry_line = ''
    if validation is not None:
        failed = validation.get('outcome_reasons', [])
        schema_failed = validation.get('failed_schema_checks', [])
        retry_line = f"\nA previous generation was mechanically rejected for: {failed}; failed schema checks: {schema_failed}. Generate a new item rather than discussing the failure."
    return user.rstrip() + f"""

MACHINE-VALIDATED OUTPUT REQUIREMENTS:
- Return exactly one item inside one top-level JSON array, with no prose, fences, or thinking tags.
- Copy archetype exactly as {arch}.
- Set period to the source's required APUSH period integer {source['period']}.
- Use one allowed theme id: NAT, WXT, GEO, MIG, PCE, WOR, ARC, or SOC.
- Include exactly four nonempty string options and answer exactly A, B, C, or D.
- Include nonempty answer_dating and requires_outside_knowledge strings.
- rationale must contain nonempty correct, A, B, C, and D strings; the keyed letter must be marked correct.
- trap_types must be a JSON array of exactly three allowed nonempty string ids.
Silently validate these mechanical requirements before returning only the JSON array.{retry_line}
"""
def inspect_contract_trace(trace, source, arch, allow_unknown_finish=False):
    raw_value = trace.get('raw'); raw = raw_value if isinstance(raw_value, str) else ''
    diagnostics = generation_format_diagnostics(raw); items = extract_items(raw); schema = None
    if diagnostics['exact_contract_valid']:
        exact_item = canonicalize_item_archetype(json.loads(raw)[0], requested_archetype=arch)
        schema = run_checks(exact_item, source, requested_archetype=arch)
    execution_reasons = []
    if not isinstance(raw_value, str): execution_reasons.append('invalid_raw_type')
    if not raw.strip(): execution_reasons.append('empty_generation')
    allowed_finish = {'json_stop', 'eos'} | ({'unknown'} if allow_unknown_finish else set())
    if trace.get('finish_reason') not in allowed_finish | {'max_new_tokens'}: execution_reasons.append('unknown_finish_reason')
    outcome_reasons = []
    if not diagnostics['no_think_tags']: outcome_reasons.append('thinking_tag')
    if len(items) == 0: outcome_reasons.append('zero_items')
    if len(items) > 1: outcome_reasons.append('multiple_items')
    if trace.get('finish_reason') == 'max_new_tokens': outcome_reasons.append('token_ceiling')
    if not diagnostics['exact_contract_valid']: outcome_reasons.append('non_exact_contract')
    if not schema or not schema['schema_valid']: outcome_reasons.append('schema_invalid')
    failed_schema_checks = [key for key in SCHEMA_VALIDATION_KEYS if schema is not None and not schema.get(key, False)]
    return {'contract_valid':not execution_reasons and not outcome_reasons,'raw':raw,'format':diagnostics,'schema':schema,
            'n_items':len(items),'execution_reasons':sorted(set(execution_reasons)),
            'outcome_reasons':sorted(set(outcome_reasons)),'failed_schema_checks':failed_schema_checks}

PREFLIGHT_PASSED = False
PREFLIGHT_TRACES = []
def run_generation_preflight():
    global PREFLIGHT_PASSED, PREFLIGHT_TRACES
    litmus_ids = splits['splits'][PREFLIGHT_SPLIT]['source_ids']
    litmus_sources = [by_id[source_id] for source_id in litmus_ids]
    if not litmus_sources:
        raise RuntimeError('Generation preflight has no sources to evaluate.')
    outcome_failures = []; execution_failures = []; protocol_failures = []
    PREFLIGHT_TRACES = []
    for model_name, tuned_flag in [('qwen3-4b-base', False), ('qwen3-apush-tuned', True)]:
        for source in litmus_sources:
            for arch in ARCHETYPES:
                system, user = prompt.build(source=source['text'], attribution=source.get('attribution', ''), note='', n=1, archetypes=arch, difficulty=DIFFICULTY, include_fewshot=INCLUDE_FEWSHOT)
                user = contract_constrained_user(user, source, arch)
                seed = attempt_seed(EVAL_SEED, 0, source['id'], arch, namespace='generation-preflight')
                trace = hf_engine.generate(system, user, tuned=tuned_flag, seed=seed, temperature=0.0, max_new_tokens=MAX_NEW_TOKENS)
                validation = inspect_contract_trace(trace, source, arch)
                outcome_reasons = validation['outcome_reasons']; execution_reasons = validation['execution_reasons']
                record = {'model':model_name,'source_id':source['id'],'archetype':arch,**trace,**validation}
                PREFLIGHT_TRACES.append(record)
                if outcome_reasons: outcome_failures.append(record)
                if execution_reasons: execution_failures.append(record)
                if 'thinking_tag' in outcome_reasons: protocol_failures.append(record)
                status = 'EXECUTION_FAIL' if execution_reasons else 'EXECUTION_PASS'
                print('preflight', model_name, source['id'], arch, trace['finish_reason'], status, 'outcomes=', outcome_reasons or ['product_valid'])
    preflight_debug_path = WORKDIR / 'preflight_generation_attempts.json'
    preflight_debug_path.write_text(json.dumps(PREFLIGHT_TRACES, indent=2, ensure_ascii=False), encoding='utf-8')
    print('preflight artifact:', preflight_debug_path)
    if execution_failures:
        first = execution_failures[0]
        raise RuntimeError(f"Generation preflight execution failed for {len(execution_failures)}/{len(PREFLIGHT_TRACES)} attempts before any API call. First: {first['model']} {first['source_id']} {first['archetype']} {first['execution_reasons']} raw={first['raw'][:300]!r}")
    if protocol_failures:
        first = protocol_failures[0]
        raise RuntimeError(f"Generation preflight protocol failed for {len(protocol_failures)}/{len(PREFLIGHT_TRACES)} attempts before any API call. First: {first['model']} {first['source_id']} {first['archetype']} {first['outcome_reasons']} raw={first['raw'][:300]!r}")
    PREFLIGHT_PASSED = True
    outcome_counts = Counter(reason for record in outcome_failures for reason in record['outcome_reasons'])
    print(f'Generation preflight execution passed: {len(PREFLIGHT_TRACES)} attempts; observed product outcomes: {dict(outcome_counts)}')

run_generation_preflight()


## Inline Eval Runner

Batches the three local HF repetitions for each `(source, archetype)` prompt.
Teacher generation and judging use bounded API concurrency.
Generation and scoring checkpoints resume automatically after an interruption.

In [ ]:
MODEL_ROSTER = [{'name':'qwen3-4b-base','role':'candidate','kind':'hf_base'}, {'name':'qwen3-apush-tuned','role':'candidate','kind':'hf_tuned'}]
REQUIRED_COMPARISON_MODELS = ['qwen3-4b-base', 'qwen3-apush-tuned']
if RUN_TEACHER: MODEL_ROSTER.append({'name':'teacher','role':'teacher','kind':'api_teacher'})
print(MODEL_ROSTER)

def generate_model(model, system, user, seed=None, max_new_tokens=None):
    token_limit = MAX_NEW_TOKENS if max_new_tokens is None else max_new_tokens
    if model['kind'] == 'hf_base':
        trace = hf_engine.generate(system, user, tuned=False, temperature=TEMPERATURE, max_new_tokens=token_limit, seed=seed)
    elif model['kind'] == 'hf_tuned':
        trace = hf_engine.generate(system, user, tuned=True, temperature=TEMPERATURE, max_new_tokens=token_limit, seed=seed)
    if model['kind'] == 'api_teacher':
        raw = api_generate(TEACHER_MODEL, system, user, max_tokens=8192, temperature=TEMPERATURE)
        return {'raw':raw,'seed':None,'generated_token_count':None,'prompt_token_count':None,'finish_reason':'unknown','max_new_tokens':8192,'rendered_prompt_sha256':stable_hash({'system':system,'user':user})}
    elif model['kind'] not in {'hf_base', 'hf_tuned'}:
        raise ValueError(model)
    trace['max_new_tokens'] = token_limit
    return trace

def generate_model_repetitions(model, system, user, seeds):
    if model['kind'] == 'hf_base': traces = hf_engine.generate_repetitions(system, user, tuned=False, seeds=seeds, temperature=TEMPERATURE, max_new_tokens=MAX_NEW_TOKENS)
    elif model['kind'] == 'hf_tuned': traces = hf_engine.generate_repetitions(system, user, tuned=True, seeds=seeds, temperature=TEMPERATURE, max_new_tokens=MAX_NEW_TOKENS)
    else:
        if len(seeds) != 1: raise ValueError('API teacher repetitions are scheduled as separate concurrent jobs')
        return [generate_model(model, system, user, seed=seeds[0])]
    for trace in traces: trace['max_new_tokens'] = MAX_NEW_TOKENS
    return traces

def score_record(rec):
    prog = run_checks(rec['item'], rec['source'])
    j = judge_item(rec['source'], rec['item']) if USE_JUDGE else None
    craft_ok = prog['disqualifying_ok'] and prog.get('craft_ok', prog.get('wrong_era_le1', True))
    judgment_ok = bool(j and j.get('_status', 'ok') == 'ok' and j.get('key_valid') is not None)
    key_valid = bool(j['key_valid'] and prog['disqualifying_ok']) if judgment_ok else None
    archetype_label_match = rec['item'].get('_model_archetype', rec['item'].get('archetype')) == rec.get('archetype')
    label_clean = bool(key_valid and prog.get('schema_ok') and prog.get('craft_ok') and archetype_label_match) if judgment_ok else None
    return {**rec, 'source_genre': source_genre(rec['source']), 'prog': prog, 'judge': j, 'near_miss': near_grade(craft_ok, j),
            'expert_grade': expert_grade(craft_ok, j), 'key_valid': key_valid,
            'label_clean': label_clean, 'distillable_item_valid': label_clean,
            'archetype_label_match': archetype_label_match}

def rate(records, key):
    vals = [r[key] for r in records if r[key] is not None]
    return None if not vals else sum(1 for v in vals if v) / len(vals)
def aggregate(records):
    n = len(records); by_arch = defaultdict(list); by_genre = defaultdict(list)
    for r in records:
        by_arch[r['item'].get('archetype', '?')].append(r)
        by_genre[r.get('source_genre', 'unknown')].append(r)
    n_judged = sum(1 for r in records if r.get('near_miss') is not None)
    return {'n_items':n, 'n_judged':n_judged, 'judge_unavailable':n-n_judged, 'expert_grade':rate(records,'expert_grade'), 'near_miss':rate(records,'near_miss'),
            'key_valid':rate(records,'key_valid'), 'label_clean':rate(records,'label_clean'), 'distillable_item_valid':rate(records,'distillable_item_valid'),
            'craft_fail':sum(1 for r in records if not r['prog'].get('craft_ok', True))/n if n else 0,
            'schema_fail':sum(1 for r in records if not r['prog'].get('schema_ok', True))/n if n else 0,
            'date_fail':sum(1 for r in records if r['prog'].get('date_direction') == 'fail')/n if n else 0,
            'leak':sum(1 for r in records if r['prog'].get('source_leak'))/n if n else 0,
            'archetype_label_fail':sum(1 for r in records if not r.get('archetype_label_match', True))/n if n else 0,
            'by_arch':{a: rate(v,'near_miss') for a,v in by_arch.items()},
            'by_genre':{g:{'n':len(v),'near_miss':rate(v,'near_miss'),'key_valid':rate(v,'key_valid')} for g,v in sorted(by_genre.items())}}
def pct(x): return 'n/a' if x is None else f'{x:.0%}'

def resolve_checkpoint_root():
    explicit = os.environ.get(CHECKPOINT_DIR_ENV)
    if explicit:
        root = Path(explicit).expanduser()
    elif PERSIST_CHECKPOINTS_TO_DRIVE and Path('/content').exists():
        try:
            from google.colab import drive
            drive.mount('/content/drive', force_remount=False)
            root = Path('/content/drive/MyDrive/apush_eval_checkpoints')
        except Exception as exc:
            print('Google Drive checkpoint mount unavailable; using runtime-local checkpoints:', repr(exc))
            root = WORKDIR / 'checkpoints'
    else:
        root = WORKDIR / 'checkpoints'
    root.mkdir(parents=True, exist_ok=True)
    return root

def eval_checkpoint_signature():
    return {'execution_version':EVAL_EXECUTION_VERSION,'split':SPLIT,'source_ids':[s['id'] for s in sources],'archetypes':ARCHETYPES,'n':N,'difficulty':DIFFICULTY,'include_fewshot':INCLUDE_FEWSHOT,
            'models':MODEL_ROSTER,'runs_by_model':{m['name']:runs_for_model(m,RUNS,TEACHER_RUNS) for m in MODEL_ROSTER},
            'base_model_id':BASE_MODEL_ID,'base_model_revision':BASE_MODEL_REVISION,'adapter_id':APUSH_ADAPTER_ID,'adapter_revision':MODEL_REVISIONS.get(APUSH_ADAPTER_ID),
            'adapter_training_data_sha256':ADAPTER_TRAINING_DATA_SHA256,'adapter_training_run_metadata_sha256':TRAINING_RUN_METADATA_SHA256,'adapter_training_run_metadata':TRAINING_RUN_METADATA,
            'temperature':TEMPERATURE,'max_new_tokens':MAX_NEW_TOKENS,'eval_seed':EVAL_SEED,'load_in_4bit':LOAD_IN_4BIT,
            'contract_retry_policy':{'mode':'first_valid','max_attempts':MAX_CONTRACT_ATTEMPTS,'feedback':'mechanical_schema_only','same_policy_for_base_and_tuned':True,'on_exhaustion':'exclude_matched_candidate_prompt_and_continue'},
            'generation_config':{'keep_no_think_prefill':True,'force_json_array_prefix':False,'use_no_think_soft_switch':False,'stop_after_complete_json_array':True,'seed_strategy':'matched independent run-indexed torch.Generator per repeated input row'},
            'teacher_model':TEACHER_MODEL if RUN_TEACHER else None,'judge_model':JUDGE_MODEL if USE_JUDGE else None,
            'repository_commit':GITHUB_REF,'canonical_notebook_sha256':CANONICAL_NOTEBOOK_SHA256,'critical_generation_cell_sha256':CRITICAL_GENERATION_CELL_SHA256,'preflight_cell_sha256':PREFLIGHT_CELL_SHA256,
            'prompt_sha256':hashlib.sha256(prompt_md.encode('utf-8')).hexdigest(),'source_corpus_sha256':hashlib.sha256(seed_stimuli_text.encode('utf-8')).hexdigest(),'split_manifest_sha256':hashlib.sha256(json.dumps(splits,sort_keys=True).encode('utf-8')).hexdigest(),'theme_vocabulary_sha256':hashlib.sha256(theme_vocabulary_text.encode('utf-8')).hexdigest(),
            'base_tokenizer_revision':BASE_MODEL_REVISION,'tokenizer_file_sha256':TOKENIZER_FILE_SHA256,'tokenizer_chat_template_sha256':CHAT_TEMPLATE_SHA256,
            'python_version':sys.version,'package_versions':PACKAGE_VERSIONS,'gpu':torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,'dtype':hf_engine.dtype_name,'quantization':'bnb-4bit-nf4' if LOAD_IN_4BIT else 'none',
            'evaluator_file_sha256':{p:hashlib.sha256((WORKDIR/p).read_bytes()).hexdigest() for p in EVALUATOR_PATHS}}

def make_attempt(model, run_idx, source, arch, trace, batch_repetitions):
    validation = inspect_contract_trace(trace, source, arch, allow_unknown_finish=model['kind'] == 'api_teacher')
    raw = validation['raw']
    return {'model':model['name'],'role':model['role'],'kind':model['kind'],'run':run_idx,'source_id':source['id'],'archetype':arch,
            'seed':trace.get('seed'),'batch_repetitions':batch_repetitions,'generated_token_count':trace.get('generated_token_count'),'max_new_tokens':trace.get('max_new_tokens'),
            'prompt_token_count':trace.get('prompt_token_count'),'finish_reason':trace.get('finish_reason','unknown'),'rendered_prompt_sha256':trace.get('rendered_prompt_sha256'),
            'n_items':validation['n_items'],'error':None,'format':validation['format'],'schema':validation['schema'],
            'contract_valid':validation['contract_valid'],'contract_reasons':validation['outcome_reasons'] + validation['execution_reasons'],
            'failed_schema_checks':validation['failed_schema_checks'],'raw_preview':raw[:1000],'raw':raw}

def contract_trial_snapshot(attempt, trial_index):
    return {'trial_index':trial_index,'seed':attempt.get('seed'),'finish_reason':attempt.get('finish_reason'),
            'generated_token_count':attempt.get('generated_token_count'),'max_new_tokens':attempt.get('max_new_tokens'),'prompt_token_count':attempt.get('prompt_token_count'),
            'rendered_prompt_sha256':attempt.get('rendered_prompt_sha256'),'n_items':attempt['n_items'],
            'contract_valid':attempt['contract_valid'],'contract_reasons':attempt['contract_reasons'],
            'failed_schema_checks':attempt['failed_schema_checks'],'format':attempt['format'],'schema':attempt['schema'],
            'raw_preview':attempt['raw_preview'],'raw':attempt['raw']}

def ensure_contract_valid_attempt(model, run_idx, source, arch, system, base_user, initial_trace, batch_repetitions):
    trace = initial_trace; trials = []
    for trial_index in range(MAX_CONTRACT_ATTEMPTS):
        attempt = make_attempt(model, run_idx, source, arch, trace, batch_repetitions if trial_index == 0 else 1)
        trials.append(contract_trial_snapshot(attempt, trial_index))
        if attempt['contract_valid']:
            break
        if trial_index + 1 >= MAX_CONTRACT_ATTEMPTS:
            break
        validation = {'outcome_reasons':attempt['contract_reasons'],'failed_schema_checks':attempt['failed_schema_checks']}
        retry_user = contract_constrained_user(base_user, source, arch, validation)
        retry_seed = attempt_seed(EVAL_SEED, run_idx, source['id'], arch, namespace=f'contract-retry-{trial_index + 1}')
        retry_token_limit = min(1536, MAX_NEW_TOKENS * 2) if 'token_ceiling' in attempt['contract_reasons'] else MAX_NEW_TOKENS
        trace = generate_model(model, system, retry_user, seed=retry_seed, max_new_tokens=retry_token_limit)
    attempt = dict(attempt)
    attempt.update({'generation_attempt_count':len(trials),'contract_retry_count':len(trials)-1,
                    'first_pass_contract_valid':trials[0]['contract_valid'],'contract_retry_exhausted':not attempt['contract_valid'],
                    'generation_trials':trials})
    return attempt

def run_eval():
    global EVAL_CHECKPOINT_INFO, JUDGING_EXCLUSIONS
    if not PREFLIGHT_PASSED: raise RuntimeError('Generation-only LITMUS preflight has not passed; API calls are blocked.')
    n_per = max(1, round(N / len(ARCHETYPES)))
    run_counts = {m['name']:runs_for_model(m, RUNS, TEACHER_RUNS) for m in MODEL_ROSTER}
    total = sum(run_counts[m['name']] for m in MODEL_ROSTER) * len(sources) * len(ARCHETYPES)
    signature = eval_checkpoint_signature(); checkpoint_id = stable_hash(signature)[:16]; checkpoint_root = resolve_checkpoint_root()
    generation_path = checkpoint_root / f'{SPLIT.lower()}_{checkpoint_id}_generation.jsonl'
    scoring_path = checkpoint_root / f'{SPLIT.lower()}_{checkpoint_id}_scoring.jsonl'
    meta_path = checkpoint_root / f'{SPLIT.lower()}_{checkpoint_id}_meta.json'
    meta_path.write_text(json.dumps(signature, indent=2), encoding='utf-8')
    checkpoint_rows, ignored_generation_lines = load_jsonl_groups(generation_path)
    attempts_by_key = {key:row for key,row in dedupe_rows(checkpoint_rows, attempt_key).items() if row.get('contract_valid') is True or row.get('contract_retry_exhausted') is True}
    expected_attempt_keys = {(m['name'],run_idx,s['id'],arch) for m in MODEL_ROSTER for run_idx in range(run_counts[m['name']]) for s in sources for arch in ARCHETYPES}
    attempts_by_key = {key:value for key,value in attempts_by_key.items() if key in expected_attempt_keys}
    initial_attempts = len(attempts_by_key); done = initial_attempts
    print(f'checkpoint: {generation_path} resumed={initial_attempts}/{total} ignored_lines={ignored_generation_lines}')

    for model in [m for m in MODEL_ROSTER if m['kind'] in ('hf_base','hf_tuned')]:
        repetitions = run_counts[model['name']]
        print('\n===', model['name'], f'(HF batches of {repetitions}) ===')
        for source in sources:
            for arch in ARCHETYPES:
                missing_runs = [run_idx for run_idx in range(repetitions) if (model['name'],run_idx,source['id'],arch) not in attempts_by_key]
                if not missing_runs:
                    continue
                system, base_user = prompt.build(source=source['text'], attribution=source.get('attribution',''), note='', n=n_per, archetypes=arch, difficulty=DIFFICULTY, include_fewshot=INCLUDE_FEWSHOT)
                user = contract_constrained_user(base_user, source, arch)
                seeds = [attempt_seed(EVAL_SEED, run_idx, source['id'], arch) for run_idx in missing_runs]
                traces = generate_model_repetitions(model, system, user, seeds)
                if len(traces) != len(missing_runs):
                    raise RuntimeError(f"Expected {len(missing_runs)} HF traces, received {len(traces)}")
                new_attempts = []
                for run_idx, trace in zip(missing_runs, traces):
                    new_attempts.append(ensure_contract_valid_attempt(model, run_idx, source, arch, system, base_user, trace, len(missing_runs)))
                append_jsonl_group(generation_path, new_attempts)
                for attempt in new_attempts:
                    attempts_by_key[attempt_key(attempt)] = attempt; done += 1
                    print(f"{done}/{total} {model['name']} run={attempt['run']} source={source['id']} arch={arch} valid={attempt['contract_valid']} raw_generations={attempt['generation_attempt_count']}")

    teacher_jobs = []
    for model in [m for m in MODEL_ROSTER if m['kind'] == 'api_teacher']:
        for run_idx in range(run_counts[model['name']]):
            for source in sources:
                for arch in ARCHETYPES:
                    if (model['name'],run_idx,source['id'],arch) not in attempts_by_key:
                        teacher_jobs.append((model,run_idx,source,arch))
    if teacher_jobs:
        print(f'\n=== teacher API: {len(teacher_jobs)} calls with {API_MAX_WORKERS} workers ===')
        def run_teacher_job(job):
            model, run_idx, source, arch = job
            system, base_user = prompt.build(source=source['text'], attribution=source.get('attribution',''), note='', n=n_per, archetypes=arch, difficulty=DIFFICULTY, include_fewshot=INCLUDE_FEWSHOT)
            user = contract_constrained_user(base_user, source, arch)
            trace = generate_model(model, system, user)
            return ensure_contract_valid_attempt(model, run_idx, source, arch, system, base_user, trace, 1)
        api_errors = []; checkpoint_buffer = []
        pool = ThreadPoolExecutor(max_workers=API_MAX_WORKERS)
        futures = {pool.submit(run_teacher_job, job):job for job in teacher_jobs}
        try:
            for future in as_completed(futures):
                job = futures[future]
                try:
                    attempt = future.result()
                except Exception as exc:
                    api_errors.append((job, repr(exc))); continue
                checkpoint_buffer.append(attempt)
                attempts_by_key[attempt_key(attempt)] = attempt; done += 1
                print(f"{done}/{total} teacher run={attempt['run']} source={attempt['source_id']} arch={attempt['archetype']} valid={attempt['contract_valid']} raw_generations={attempt['generation_attempt_count']}")
                if len(checkpoint_buffer) >= CHECKPOINT_FLUSH_EVERY:
                    append_jsonl_group(generation_path, checkpoint_buffer); checkpoint_buffer = []
        finally:
            for future in futures: future.cancel()
            pool.shutdown(wait=True, cancel_futures=True)
            append_jsonl_group(generation_path, checkpoint_buffer)
        if api_errors:
            raise RuntimeError(f'Teacher API failed for {len(api_errors)} calls; completed calls are checkpointed. First failure: {api_errors[0]}')

    missing_attempts = sorted(expected_attempt_keys - set(attempts_by_key))
    if missing_attempts: raise RuntimeError(f'Generation is incomplete for {len(missing_attempts)} logical prompts. First: {missing_attempts[0]}')
    model_order = {m['name']:idx for idx,m in enumerate(MODEL_ROSTER)}; source_order = {s['id']:idx for idx,s in enumerate(sources)}; arch_order = {a:idx for idx,a in enumerate(ARCHETYPES)}
    attempts = sorted(attempts_by_key.values(), key=lambda a:(model_order[a['model']],a['run'],source_order[a['source_id']],arch_order[a['archetype']]))
    candidate_exclusions = matched_candidate_exclusions(attempts, REQUIRED_COMPARISON_MODELS)
    annotated_attempts = []; JUDGING_EXCLUSIONS = []
    for attempt in attempts:
        attempt = dict(attempt); logical_key = (attempt['run'], attempt['source_id'], attempt['archetype'])
        exclusion = candidate_exclusions.get(logical_key) if attempt['model'] in REQUIRED_COMPARISON_MODELS else None
        if exclusion is None and attempt['contract_valid'] is not True:
            exclusion = {'reason':'model_contract_exhaustion','invalid_models':[attempt['model']]}
        attempt['judging_excluded'] = exclusion is not None
        attempt['judging_exclusion'] = exclusion
        annotated_attempts.append(attempt)
        if exclusion is not None:
            JUDGING_EXCLUSIONS.append({'model':attempt['model'],'run':attempt['run'],'source_id':attempt['source_id'],'archetype':attempt['archetype'],**exclusion})
    attempts = annotated_attempts
    print(f'Judging exclusions: {len(JUDGING_EXCLUSIONS)} model outputs across {len(candidate_exclusions)} matched candidate prompts')
    source_by_id = {s['id']:s for s in sources}; records = []
    for attempt in attempts:
        if attempt['judging_excluded']: continue
        source = source_by_id[attempt['source_id']]
        for item_index, item in enumerate(extract_items(attempt['raw'])):
            item = canonicalize_item_archetype(item, requested_archetype=attempt['archetype'])
            records.append({'model':attempt['model'],'role':attempt['role'],'run':attempt['run'],'source_id':attempt['source_id'],'source':source,'archetype':attempt['archetype'],'item_index':item_index,'item_sha256':stable_hash(item),'item':item,'raw':attempt['raw'][:5000],'error':attempt.get('error')})

    score_rows, ignored_score_lines = load_jsonl_groups(scoring_path); scored_by_key = dedupe_rows(score_rows, score_key)
    expected_score_keys = {score_key(record) for record in records}; scored_by_key = {key:value for key,value in scored_by_key.items() if key in expected_score_keys}
    initial_scores = len(scored_by_key); print(f'\nScoring: resumed={initial_scores}/{len(records)} ignored_lines={ignored_score_lines} workers={API_MAX_WORKERS if USE_JUDGE else 1}')
    records_to_score = [record for record in records if score_key(record) not in scored_by_key]
    score_errors = []; checkpoint_buffer = []; score_done = initial_scores
    pool = ThreadPoolExecutor(max_workers=API_MAX_WORKERS if USE_JUDGE else 1)
    futures = {pool.submit(score_record, record):record for record in records_to_score}
    try:
        for future in as_completed(futures):
            record = futures[future]
            try:
                scored_record = future.result()
            except Exception as exc:
                score_errors.append((score_key(record), repr(exc))); continue
            slim = {key:value for key,value in scored_record.items() if key != 'source'}
            scored_by_key[score_key(slim)] = slim; checkpoint_buffer.append(slim); score_done += 1
            if score_done % 5 == 0 or score_done == len(records): print(f'scored {score_done}/{len(records)}')
            if len(checkpoint_buffer) >= CHECKPOINT_FLUSH_EVERY:
                append_jsonl_group(scoring_path, checkpoint_buffer); checkpoint_buffer = []
    finally:
        for future in futures: future.cancel()
        pool.shutdown(wait=True, cancel_futures=True)
        append_jsonl_group(scoring_path, checkpoint_buffer)
    if score_errors:
        raise RuntimeError(f'Judging failed for {len(score_errors)} items; completed scores are checkpointed. First failure: {score_errors[0]}')
    scored = [{**scored_by_key[score_key(record)],'source':source_by_id[record['source_id']]} for record in records]
    EVAL_CHECKPOINT_INFO = {'id':checkpoint_id,'generation_path':str(generation_path),'scoring_path':str(scoring_path),'resumed_generation_attempts':initial_attempts,'resumed_scored_items':initial_scores,'judging_exclusions':len(JUDGING_EXCLUSIONS)}
    return scored, attempts

scored, generation_attempts = run_eval()
print('records:', len(scored))

## Results

In [ ]:
roster_names = [m['name'] for m in MODEL_ROSTER]
attempt_names = sorted({a['model'] for a in generation_attempts})
scored_names = sorted({r['model'] for r in scored})
model_names = []
for name in roster_names + attempt_names + scored_names:
    if name not in model_names:
        model_names.append(name)

missing_attempts = [m for m in REQUIRED_COMPARISON_MODELS if m not in attempt_names]
if missing_attempts:
    raise RuntimeError(f"Base-vs-tuned eval is invalid: missing generation attempts for {missing_attempts}. Rerun the Inline Eval Runner cell from a fresh runtime.")
expected_calls_per_model = RUNS * len(sources) * len(ARCHETYPES)
bad_call_counts = {model:sum(a['model'] == model for a in generation_attempts) for model in REQUIRED_COMPARISON_MODELS if sum(a['model'] == model for a in generation_attempts) != expected_calls_per_model}
if bad_call_counts: raise RuntimeError(f'Base-vs-tuned eval is incomplete: expected {expected_calls_per_model} calls per model, found {bad_call_counts}')

def prompt_key(record): return (record['run'], record['source_id'], record['archetype'])

results = {}
for model in model_names:
    model_scored = [r for r in scored if r['model'] == model]
    results[model] = aggregate(model_scored)
    calls = [a for a in generation_attempts if a['model'] == model]
    judge_eligible_calls = [a for a in calls if not a.get('judging_excluded')]
    parsed = results[model]['n_items']
    zero_calls = sum(1 for a in calls if a['n_items'] == 0)
    generation = aggregate_attempt_metrics(judge_eligible_calls, model_scored)
    raw_generations = sum(a.get('generation_attempt_count', 1) for a in calls)
    first_pass_valid = sum(bool(a.get('first_pass_contract_valid', a.get('contract_valid'))) for a in calls)
    valid_outputs = sum(a.get('contract_valid') is True for a in calls)
    generation.update({'calls':len(calls),'judging_eligible_calls':len(judge_eligible_calls),'judging_excluded_calls':len(calls)-len(judge_eligible_calls),
                       'zero_item_calls':zero_calls,'parse_empty_rate':zero_calls/len(calls) if calls else None,'parsed_items_per_call':parsed/len(judge_eligible_calls) if judge_eligible_calls else None,
                       'judge_inconclusive_calls':generation['judge_unavailable_attempts'],'resolved_attempts':generation['successfully_judged_attempts'],
                       'attempted_prompt_near_miss':generation['attempted_prompt_near_miss_rate'],'near_miss_per_resolved_attempt':generation['near_miss_per_successfully_judged_attempt'],
                       'raw_generation_attempts':raw_generations,'first_pass_contract_valid_rate':first_pass_valid/len(calls) if calls else None,
                       'mean_generations_per_valid_output':raw_generations/valid_outputs if valid_outputs else None,
                       'retried_logical_prompts':sum(a.get('contract_retry_count', 0) > 0 for a in calls),
                       'contract_retry_exhausted':sum(bool(a.get('contract_retry_exhausted')) for a in calls)})
    results[model]['generation'] = generation

def fmt_num(x): return 'n/a' if x is None else f'{x:.2f}'
print('| Model | Calls | Excluded | Zero | Parsed | Judged | Judge unavailable | Near/attempt | Strict arrays | Exact contract | Schema/attempt | Product contract | Expert | Near/judged | Key valid | Label clean | Craft fail | Schema fail | Arch label fail | Date fail | Leak |')
print('| :--- | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: |')
for model in model_names:
    agg = results[model]
    gen = agg['generation']
    print(f"| {model} | {gen['calls']} | {gen['judging_excluded_calls']} | {gen['zero_item_calls']} | {agg['n_items']} | {agg['n_judged']} | {gen['judge_unavailable_attempts']} | {pct(gen['attempted_prompt_near_miss_rate'])} | {pct(gen['strict_array_rate'])} | {pct(gen['exact_contract_rate'])} | {pct(gen['schema_valid_rate'])} | {pct(gen['product_contract_rate'])} | {pct(agg['expert_grade'])} | {pct(agg['near_miss'])} | {pct(agg['key_valid'])} | {pct(agg['label_clean'])} | {pct(agg['craft_fail'])} | {pct(agg['schema_fail'])} | {pct(agg['archetype_label_fail'])} | {pct(agg['date_fail'])} | {pct(agg['leak'])} |")
    print('  format buckets:', gen['format_buckets'])
print('\nQuality metrics above are conditional on the first contract-valid output selected by the shared retry policy; matched exhausted prompts are excluded from both candidate denominators.')
print('| Model | Logical prompts | Raw generations | First-pass valid | Retried prompts | Mean generations/valid | Exhausted | Excluded from judging |')
print('| :--- | ---: | ---: | ---: | ---: | ---: | ---: | ---: |')
for model in model_names:
    gen = results[model]['generation']
    print(f"| {model} | {gen['calls']} | {gen['raw_generation_attempts']} | {pct(gen['first_pass_contract_valid_rate'])} | {gen['retried_logical_prompts']} | {fmt_num(gen['mean_generations_per_valid_output'])} | {gen['contract_retry_exhausted']} | {gen['judging_excluded_calls']} |")

print('\n| Model | Source genre | Items | Near | Key valid |')
print('| :--- | :--- | ---: | ---: | ---: |')
for model in model_names:
    for genre, stats in results[model].get('by_genre', {}).items():
        print(f"| {model} | {genre} | {stats['n']} | {pct(stats['near_miss'])} | {pct(stats['key_valid'])} |")

comparison = {'base_model': 'qwen3-4b-base', 'tuned_model': 'qwen3-apush-tuned'}
base = results.get(comparison['base_model'], {})
tuned = results.get(comparison['tuned_model'], {})
for metric in ['near_miss', 'key_valid', 'label_clean', 'distillable_item_valid']:
    b = base.get(metric); t = tuned.get(metric)
    comparison[f'{metric}_delta_tuned_minus_base'] = None if b is None or t is None else t - b
results['_comparison'] = comparison
print('\nBase-vs-tuned comparison:', json.dumps(comparison, indent=2))

def exact_mcnemar_p(base_rows, tuned_rows, metric):
    key = lambda r: (r['run'], r['source_id'], r['archetype'])
    bmap = {key(r): bool(r.get(metric)) for r in base_rows if r.get(metric) is not None}; tmap = {key(r): bool(r.get(metric)) for r in tuned_rows if r.get(metric) is not None}
    paired = sorted(set(bmap) & set(tmap)); improve = sum((not bmap[k]) and tmap[k] for k in paired); regress = sum(bmap[k] and (not tmap[k]) for k in paired)
    n = improve + regress
    p = 1.0 if n == 0 else min(1.0, 2 * sum(math.comb(n, i) for i in range(min(improve, regress) + 1)) / (2 ** n))
    return {'paired_prompts':len(paired),'base_fail_tuned_pass':improve,'base_pass_tuned_fail':regress,'exact_mcnemar_p':p}
def source_clustered_paired_test(base_rows, tuned_rows, metric, draws=10000):
    key = lambda r: (r['run'], r['source_id'], r['archetype'])
    bmap = {key(r): int(bool(r.get(metric))) for r in base_rows if r.get(metric) is not None}
    tmap = {key(r): int(bool(r.get(metric))) for r in tuned_rows if r.get(metric) is not None}
    paired = sorted(set(bmap) & set(tmap)); by_source = defaultdict(list)
    for pair in paired: by_source[pair[1]].append(tmap[pair] - bmap[pair])
    source_deltas = [sum(values) / len(values) for values in by_source.values()]
    if not source_deltas: return {'sources':0,'mean_delta':None,'bootstrap_95_ci':None,'sign_flip_p':None}
    observed = sum(source_deltas) / len(source_deltas); rng = random.Random(42); n = len(source_deltas)
    boot = sorted(sum(rng.choice(source_deltas) for _ in range(n)) / n for _ in range(draws))
    lo = boot[int(0.025 * draws)]; hi = boot[min(draws - 1, int(0.975 * draws))]
    extreme = 0
    for _ in range(draws):
        permuted = sum(delta * (1 if rng.random() < 0.5 else -1) for delta in source_deltas) / n
        extreme += abs(permuted) >= abs(observed)
    return {'sources':n,'mean_delta':observed,'bootstrap_95_ci':[lo,hi],'sign_flip_p':(extreme+1)/(draws+1)}
def attempt_outcome_map(model_name, metric):
    rows_by_prompt = defaultdict(list)
    for row in scored:
        if row['model'] == model_name: rows_by_prompt[prompt_key(row)].append(row)
    outcomes = {}
    for attempt in generation_attempts:
        if attempt['model'] != model_name: continue
        key = prompt_key(attempt)
        if attempt.get('judging_excluded') and metric in {'near_miss','key_valid','label_clean'}:
            outcomes[key] = None
        else:
            outcomes[key] = attempt_contract_outcome(attempt, rows_by_prompt.get(key, []))[metric]
    return outcomes
def exact_mcnemar_maps(base_map, tuned_map):
    shared = sorted(set(base_map) & set(tuned_map)); paired = [key for key in shared if base_map[key] is not None and tuned_map[key] is not None]
    improve = sum((not base_map[key]) and tuned_map[key] for key in paired); regress = sum(base_map[key] and (not tuned_map[key]) for key in paired); n = improve + regress
    p = 1.0 if n == 0 else min(1.0, 2 * sum(math.comb(n, i) for i in range(min(improve, regress) + 1)) / (2 ** n))
    return {'shared_prompts':len(shared),'paired_prompts':len(paired),'inconclusive_excluded':len(shared)-len(paired),'base_fail_tuned_pass':improve,'base_pass_tuned_fail':regress,'exact_mcnemar_p':p}
def source_clustered_maps(base_map, tuned_map, draws=10000):
    return source_clustered_paired_ci(base_map, tuned_map, draws=draws)
base_rows=[r for r in scored if r['model']=='qwen3-4b-base']; tuned_rows=[r for r in scored if r['model']=='qwen3-apush-tuned']
comparison['paired_tests']={m:exact_mcnemar_p(base_rows,tuned_rows,m) for m in ['near_miss','key_valid','label_clean','distillable_item_valid']}
comparison['source_clustered_tests']={m:source_clustered_paired_test(base_rows,tuned_rows,m) for m in ['near_miss','key_valid','label_clean','distillable_item_valid']}
attempt_metrics = ['parse_success','exact_contract','schema_valid','product_contract_valid','near_miss','key_valid','label_clean']
comparison['attempt_level_paired_tests'] = {}
for metric in attempt_metrics:
    base_map = attempt_outcome_map('qwen3-4b-base', metric); tuned_map = attempt_outcome_map('qwen3-apush-tuned', metric)
    comparison['attempt_level_paired_tests'][metric] = {'mcnemar':exact_mcnemar_maps(base_map,tuned_map),'source_clustered':source_clustered_maps(base_map,tuned_map)}
print('Paired tests:', json.dumps(comparison['paired_tests'], indent=2))
print('Source-clustered tests:', json.dumps(comparison['source_clustered_tests'], indent=2))
print('Attempt-level paired tests:', json.dumps(comparison['attempt_level_paired_tests'], indent=2))

def failure_buckets(records):
    counts = defaultdict(int)
    for r in records:
        j=r.get('judge') or {}; p=r.get('prog') or {}
        judgment_ok = bool(j and j.get('_status', 'ok') == 'ok')
        if j and not judgment_ok: counts['judge_unavailable'] += 1
        if judgment_ok and r.get('key_valid') is False:
            if not j.get('key_historically_correct'): counts['key_factually_wrong'] += 1
            if not j.get('key_uniquely_best'): counts['key_not_unique'] += 1
        if judgment_ok and r.get('near_miss') is False:
            if not j.get('requires_outside_knowledge'): counts['no_outside_knowledge'] += 1
            if not j.get('skill_matches_command_phrase'): counts['skill_mismatch'] += 1
            if not j.get('single_best_answer'): counts['not_single_best'] += 1
            if not j.get('every_distractor_named_trap'): counts['distractor_trap_invalid'] += 1
            if not j.get('distractors_period_plausible'): counts['distractor_period_implausible'] += 1
        if not p.get('craft_ok', True): counts['programmatic_craft_fail'] += 1
        if not p.get('schema_ok', True): counts['schema_invalid'] += 1
    return dict(sorted(counts.items(), key=lambda kv:(-kv[1],kv[0])))
failure_summary={m:failure_buckets([r for r in scored if r['model']==m]) for m in model_names}
print('Failure buckets:', json.dumps(failure_summary, indent=2))

print('\nZero-item generation calls:')
zero_attempts = [a for a in generation_attempts if a['n_items'] == 0]
for a in zero_attempts[:20]:
    print(f"- {a.get('model')} {a.get('source_id', '?')} {a.get('archetype', '?')} error={a.get('error')} raw={str(a.get('raw_preview', ''))[:180]!r}")
if len(zero_attempts) > 20:
    print(f"... {len(zero_attempts) - 20} more")

out_dir = WORKDIR / 'results'; out_dir.mkdir(exist_ok=True)
preflight_path = out_dir / 'preflight_generation_attempts.json'
preflight_path.write_text(json.dumps(PREFLIGHT_TRACES, indent=2, ensure_ascii=False), encoding='utf-8')
run_meta = {**eval_checkpoint_signature(),'timestamp':datetime.now(timezone.utc).isoformat(),'runs':RUNS,'teacher_runs':TEACHER_RUNS if RUN_TEACHER else 0,'representative_source_count':len(sources),
            'expected_attempts_per_model':RUNS*len(sources)*len(ARCHETYPES),'expected_attempts_by_model':{m['name']:runs_for_model(m,RUNS,TEACHER_RUNS)*len(sources)*len(ARCHETYPES) for m in MODEL_ROSTER},
            'completed_attempts_by_model':{m:results[m]['generation']['calls'] for m in model_names},'judging_excluded_attempts_by_model':{m:results[m]['generation']['judging_excluded_calls'] for m in model_names},'parsed_attempts_by_model':{m:results[m]['generation']['parsed_attempts'] for m in model_names},
            'exact_contract_attempts_by_model':{m:results[m]['generation']['exact_contract_successes'] for m in model_names},'successfully_judged_attempts_by_model':{m:results[m]['generation']['successfully_judged_attempts'] for m in model_names},
            'attempt_seeds':[{'model':a['model'],'run':a['run'],'source_id':a['source_id'],'archetype':a['archetype'],'seed':a.get('seed')} for a in generation_attempts],
            'contract_retry_summary':{m:{key:results[m]['generation'][key] for key in ['raw_generation_attempts','first_pass_contract_valid_rate','mean_generations_per_valid_output','retried_logical_prompts','contract_retry_exhausted']} for m in model_names},
            'rendered_prompt_hashes':[{'model':a['model'],'run':a['run'],'source_id':a['source_id'],'archetype':a['archetype'],'sha256':a.get('rendered_prompt_sha256')} for a in generation_attempts],
            'preflight':{'passed':PREFLIGHT_PASSED,'attempts':len(PREFLIGHT_TRACES),'artifact':'preflight_generation_attempts.json','artifact_sha256':hashlib.sha256(preflight_path.read_bytes()).hexdigest()},
            'judging_exclusions':JUDGING_EXCLUSIONS,'api_max_workers':API_MAX_WORKERS,'checkpoint':EVAL_CHECKPOINT_INFO,'failure_buckets':failure_summary}
(out_dir / 'summary.json').write_text(json.dumps({'meta':run_meta,'results':results}, indent=2), encoding='utf-8')
(out_dir / 'generation_attempts.json').write_text(json.dumps(generation_attempts, indent=2, ensure_ascii=False), encoding='utf-8')
with (out_dir / 'items.jsonl').open('w', encoding='utf-8') as f:
    for r in scored:
        slim = {k:v for k,v in r.items() if k not in ('source','item')}; slim['item'] = dict(r['item'])
        f.write(json.dumps(slim, ensure_ascii=False) + '\n')
print('wrote:', out_dir)
import shutil
archive_path = shutil.make_archive(str(WORKDIR / 'apush_eval_results'), 'zip', root_dir=out_dir)
print('results archive:', archive_path)
if DOWNLOAD_RESULTS_ZIP:
    try:
        from google.colab import files
        files.download(archive_path)
    except Exception as exc:
        print('automatic download unavailable; download manually:', archive_path, repr(exc))


## Full Run Settings

The notebook is configured for two low-temperature matched repetitions:

```python
RUNS = 2
TEACHER_RUNS = 1
REPRESENTATIVE_SOURCE_COUNT = 14
N = 2  # one call for each of the two archetypes
MAX_NEW_TOKENS = 768  # start here; raise only if outputs are cut off
MAX_CONTRACT_ATTEMPTS = 8  # first output plus mechanical-only retries
```

This produces 56 contract-valid scored outputs per candidate model and 28 teacher reference outputs, for 140 logical scored outputs total.
The 40-attempt local preflight plus the first generation for each logical output creates a minimum of 180 raw generations; mechanical failures add bounded retries.
The two local repetitions for a prompt are generated together so the L4 can process a batch instead of repeating batch-size-one inference.
The low nonzero temperature makes candidate repetitions informative over a frozen 14-source cohort that preserves all six held-out source genres and broad chronological coverage.
Base and tuned use the same deterministic retry seeds and receive only mechanical schema feedback, never answer or judge feedback.
Expert-grade comparisons are conditional on the first contract-valid output, while first-pass validity and retry burden remain separately reported.
If either candidate exhausts all retries, that exact run-source-archetype prompt is excluded from judging for both candidates and the remaining run continues.
Teacher generation and judge scoring use bounded API concurrency.
Google Drive checkpoints resume both generation and scoring after an interruption.
The validated generation protocol is hardcoded in this notebook.
Run any inference-protocol ablation in a separate generation-only notebook so it cannot contaminate this matched semantic evaluation.

Run the notebook from the top after setting a full immutable `GITHUB_REF`; it fetches canonical shared checks and runs the complete 40-attempt LITMUS execution/protocol preflight before any API calls.
The first HF generation downloads and loads weights; subsequent cells reuse the model.
On L4, keep `LOAD_IN_4BIT=False` unless you OOM.